In [ ]:
import os
import sys
from pathlib import Path
from scipy.linalg import cho_factor, cho_solve, LinAlgError
from jax.scipy.sparse.linalg import cg
from pathlib import Path
import csv

from PIL import ImageOps
from tqdm.auto import tqdm


UPSTREAM_DIR = Path.home() / "work" / "drifting"
CACHE_DIR = Path.home() / "work" / "hf_cache"

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["HF_ROOT"] = str(CACHE_DIR)

if str(UPSTREAM_DIR) not in sys.path:
    sys.path.insert(0, str(UPSTREAM_DIR))


import numpy as np
import jax
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt
from PIL import Image

from utils.init_util import load_generator_model_and_params

print("JAX:", jax.__version__)
print("Backend:", jax.default_backend())
print("Devices:", jax.devices())
assert jax.default_backend() == "gpu"

## Stage 1 — Load the pretrained generator

We load the published generator and its parameters. During Phase 1, these parameters stay fixed; later we will optimize only the generator's noise input.

In [ ]:
model, params, metadata = load_generator_model_and_params(
    "hf://pixel_B_sota",
    hf_cache_dir=str(CACHE_DIR),
)

leaves = jax.tree_util.tree_leaves(params)
parameter_count = sum(x.size for x in leaves)

print("Parameter leaves:", len(leaves))
print("Parameters:", parameter_count)
print("Image size:", model.input_size)
print("Channels:", model.in_channels)
print("Uses bfloat16:", model.use_bf16)

### Dataset loading - ImageNet-V2-MatchedFrequencies

In [ ]:
import tarfile
from huggingface_hub import hf_hub_download

DATA_DIR = Path.home() / "work" / "datasets" / "ImageNetV2"
DATA_DIR.mkdir(parents=True, exist_ok=True)

archive_path = Path(
    hf_hub_download(
        repo_id="vaishaal/ImageNetV2",
        repo_type="dataset",
        filename="imagenetv2-matched-frequency.tar.gz",
        local_dir=DATA_DIR,
    )
)

def find_images():
    return sorted(
        path for path in DATA_DIR.rglob("*")
        if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )

image_paths = find_images()
if len(image_paths) != 10_000:
    with tarfile.open(archive_path, "r:*") as archive:
        archive.extractall(DATA_DIR, filter="data")
    image_paths = find_images()

assert len(image_paths) == 10_000, f"Found {len(image_paths)} images"
print("ImageNet-V2 images:", len(image_paths))
print("Example path:", image_paths[0])

#### Data degredation 


In [ ]:
# Image degreadation 

# Blurring
def circular_convolve2d(img, kernel):
    m, n = img.shape
    mk, nk = kernel.shape
    big_K = np.zeros((m, n))
    big_K[:mk, :nk] = kernel
    big_K = np.roll(big_K, shift=(-int((mk - 1) / 2), -int((nk - 1) / 2)), axis=(0, 1))
    F_k = np.fft.fft2(big_K)
    return np.real(np.fft.ifft2(np.fft.fft2(img) * F_k))


# Inpainting
#create a mask with a given ratio of ones to zeros
def mask_creation(image_shape, rng, ratio = 0.2):
    if ratio<=0 or ratio >1:
        raise ValueError("Ratio should be in the semi-open range (0,1]")

    if len(image_shape) != 2:
        raise ValueError("2D image is expected, image size should be a 2 element touple")
    
    # Total number of elements
    element_num = np.prod(image_shape)

    # Total number of ones
    num_ones = int(element_num * ratio)

    # Creating a 1D vector of zeros
    mask = np.zeros(shape = (element_num), dtype = int)

    # Filling the first elements to be 1
    mask[:num_ones] = 1

    # Randomly shuffling the mask
    rng.shuffle(mask)

    mask = np.reshape(mask, shape = image_shape)

    return mask



#median inpainting function for initializtion
def median_inpainting_2d(y, mask):
    # Based on Tirer-lab implementation.

    # Create a copy and convert to float to allow np.nan assignment
    y0 = y.copy().astype(float)
    
    #where mask is 0, set the pixel to NaN
    y0[mask == 0] = np.nan

    win_size = 1
    M, N = y0.shape

    # Creating a NaN map
    bitmap_NaN = np.isnan(y0)

    # Suppress warnings for all-NaN slices when the window is small
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)

        while np.count_nonzero(bitmap_NaN) > 0:
            y0_prev = y0.copy()
            win_size += 1
            rows, cols = np.where(bitmap_NaN)

            for r, c in zip(rows, cols):
                # Define the boundaries of the local patch
                row_start = max(0, r - win_size)
                row_end = min(M, r + win_size + 1)
                col_start = max(0, c - win_size)
                col_end = min(N, c + win_size + 1)

                # Extract the 2D patch around the missing pixel
                patch = y0_prev[row_start:row_end, col_start:col_end]
                
                # Calculate the median of the valid pixels
                # np.nanmedian automatically computes over the flattened array when axis=None
                y0[r, c] = np.nanmedian(patch)

            # Re-evaluate which pixels are still missing
            bitmap_NaN = np.isnan(y0)

    return y0


# Downsampling 
def A_sr4(x):
    """Antialiased bicubic: [batch, 256, 256, 3] -> [batch, 64, 64, 3]."""
    assert x.ndim == 4 and x.shape[1:] == (256, 256, 3)
    return jax.image.resize(
        x.astype(jnp.float32),
        (x.shape[0], 64, 64, 3),
        method="cubic",
        antialias=True,
    )

In [ ]:



def load_hr256(path):
    """Load an image as a centered 256×256 RGB array in [-1, 1]."""
    with Image.open(path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")

    width, height = image.size
    if width <= height:
        new_size = (256, round(height * 256 / width))
    else:
        new_size = (round(width * 256 / height), 256)

    image = image.resize(new_size, Image.Resampling.BICUBIC)
    left = (image.width - 256) // 2
    top = (image.height - 256) // 2
    image = image.crop((left, top, left + 256, top + 256))

    return np.asarray(image, dtype=np.float32) / 127.5 - 1.0


paths = sorted(Path(path) for path in image_paths)
assert len(paths) == 10_000

output_path = Path(DATA_DIR) / "imageNet_x4.npy"
index_path = Path(DATA_DIR) / "imageNet_x4_index.csv"

if not output_path.exists():
    temporary_path = Path(DATA_DIR) / "imageNet_x4.partial.npy"
    imageNet_x4_file = np.lib.format.open_memmap(
        temporary_path,
        mode="w+",
        dtype=np.float32,
        shape=(len(paths), 64, 64, 3),
    )

    downsample_batch = jax.jit(A_sr4)
    batch_size = 32

    for start in tqdm(range(0, len(paths), batch_size), desc="Creating imageNet_x4"):
        batch_paths = paths[start:start + batch_size]
        hr_batch = np.stack([load_hr256(path) for path in batch_paths])
        lr_batch = downsample_batch(jnp.asarray(hr_batch))
        imageNet_x4_file[start:start + len(batch_paths)] = np.asarray(
            jax.device_get(lr_batch), dtype=np.float32
        )

    imageNet_x4_file.flush()
    del imageNet_x4_file

    with index_path.open("w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["index", "relative_path", "label"])
        for index, path in enumerate(paths):
            writer.writerow([index, path.relative_to(DATA_DIR), int(path.parent.name)])

    temporary_path.replace(output_path)

imageNet_x4 = np.load(output_path, mmap_mode="r")
assert imageNet_x4.shape == (10_000, 64, 64, 3)
print("imageNet_x4:", imageNet_x4.shape, imageNet_x4.dtype)
print("Saved to:", output_path)

In [ ]:
def psnr(original, reconstructed, *, value_range=(-1.0, 1.0)):
    """Compute PSNR after scaling both images to the reference [0, 1] range."""
    original = np.asarray(original, dtype=np.float64)
    reconstructed = np.asarray(reconstructed, dtype=np.float64)
    low, high = value_range

    if original.shape != reconstructed.shape or original.size == 0:
        raise ValueError("Images must have the same nonempty shape.")
    if not np.isfinite(original).all() or not np.isfinite(reconstructed).all():
        raise ValueError("Images must contain only finite values.")
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        raise ValueError("value_range must be finite with high > low.")

    original_01 = (original - low) / (high - low)
    reconstructed_01 = (reconstructed - low) / (high - low)

    mse = np.mean((original_01 - reconstructed_01) ** 2)
    return float("inf") if mse == 0 else float(-10 * np.log10(mse))

### Step 1 - Pixel space optimization 

$$ \hat x= f_{\theta}(\epsilon) \Rightarrow \hat \epsilon = argmin_{\epsilon} ||y-Af_{\theta}(\epsilon)||^2$$


For this part, the model's weights, 
$\theta$
are fixed and only the input noise is being optimized
<br>
Notice that the dataset were gonna use is imageNet-V2 
On one hand this is to avoid using images that the generator was trained on, on the other, this is to use images that are semantically similar enough.

In [ ]:

# Psudo-inverse function for backprojection -> applicable only for cases where A is known
def pseudo_inverse(A, delta=0.0, rcond=1e-7):
    """
    Explicit (regularized) pseudo-inverse of A.
      wide (m <= n):  Aᵀ (AAᵀ + δI)⁻¹
      tall (m >  n):  (AᵀA + δI)⁻¹ Aᵀ
    Uses Cholesky on the smaller Gram matrix. Falls back to an SVD
    if A is rank-deficient or badly conditioned.
    """
    A = np.asarray(A, dtype=np.float64)
    m, n = A.shape
    wide = m <= n

    # Gram matrix of the *smaller* dimension: min(m,n) x min(m,n)
    G = A @ A.T if wide else A.T @ A
    if delta:
        G[np.diag_indices_from(G)] += delta          # in place, no np.eye

    try:
        c, low = cho_factor(G, overwrite_a=True, check_finite=False)
        # cheap conditioning check: cond(G) ≈ (max/min of chol diagonal)²
        d = np.abs(np.diag(c))
        if d.min() < np.sqrt(rcond) * d.max():
            raise LinAlgError("ill-conditioned")
        if wide:
            return cho_solve((c, low), A, check_finite=False).T   # = Aᵀ G⁻¹
        return cho_solve((c, low), A.T, check_finite=False)       # = G⁻¹ Aᵀ

    except LinAlgError:
        # rank-deficient or ill-conditioned: SVD is the robust route
        if delta:
            U, s, Vt = np.linalg.svd(A, full_matrices=False)
            return (Vt.T * (s / (s**2 + delta))) @ U.T
        return np.linalg.pinv(A, rcond=rcond)

In [ ]:
# Applicable only for cases where A is known
def BP(observed, estimated, deg_op):
    """Compute the backprojection of the observed image onto the estimated image."""

    A_deg = pseudo_inverse(deg_op, delta=1e-6)

    return A_deg @ (observed - deg_op @ estimated) + estimated

# Operator-free BP implementation using JAX native operations

def backproject_sr4(x_hat, y, delta=0.0, tol=1e-5, maxiter=200):
    """Correct an HR estimate using its fixed 64×64 measurement."""
    x_hat = jnp.asarray(x_hat, dtype=jnp.float32)
    y = jnp.asarray(y, dtype=jnp.float32)

    assert x_hat.ndim == 4 and x_hat.shape[1:] == (256, 256, 3)
    assert y.shape == (x_hat.shape[0], 64, 64, 3)

    # Apply Aᵀ without constructing A.
    transpose = jax.linear_transpose(A_sr4, jnp.zeros_like(x_hat))
    def AT(v):
        return transpose(v)[0]

    residual = y - A_sr4(x_hat)

    # Solve (AAᵀ + δI)u = residual.
    def normal_operator(v):
        return A_sr4(AT(v)) + delta * v

    u, _ = cg(normal_operator, residual, tol=tol, maxiter=maxiter)
    return x_hat + AT(u)

### Blurring & DeBlurring


In [ ]:
# blurring kernel
blurring_kernel = np.zeros((15, 15))
range_ij = np.linspace(-7, 7, num = 15, dtype = int)
shift_bias = 7 # To make sure indices are correct
for i in range_ij:
    for j in range_ij:
        blurring_kernel[i+shift_bias, j+shift_bias] = 1/(1+i**2 + j**2)

blurring_kernel /= np.sum(blurring_kernel)

### Inpainting

In [ ]:
# Generate a list of masks
#masks = [mask_creation(img.shape, rng = rng) for img in clean_images]

# Multiply each image by its corresponding mask
#sparse_images = [img * mask for img, mask in zip(clean_images, masks)]

#idx = 0
#median_inpainted = median_inpainting_2d(sparse_images[idx], masks[idx])


# CSGM - like approach

In [ ]:
# Regularization
def eps_regularizer(eps, kind):
    """Return a scalar penalty. `kind` must be fixed when JIT compiling."""
    e = jnp.asarray(eps, dtype=jnp.float32).reshape(-1)

    if kind == "none":
        return jnp.array(0.0, dtype=e.dtype)

    if kind == "l2":
        return 0.5 * jnp.mean(e**2)

    mean = jnp.mean(e)
    variance = jnp.mean((e - mean) ** 2)
    z = (e - mean) / jnp.sqrt(variance + 1e-12)

    if kind == "moments":
        skewness = jnp.mean(z**3)
        excess_kurtosis = jnp.mean(z**4) - 3.0
        return skewness**2 + 0.25 * excess_kurtosis**2

    if kind == "quantiles":
        n = e.size
        probabilities = (jnp.arange(n, dtype=e.dtype) + 0.5) / n
        normal_quantiles = jax.scipy.special.ndtri(probabilities)
        return jnp.mean((jnp.sort(z) - normal_quantiles) ** 2)

    raise ValueError(f"Unknown regularizer: {kind}")

In [ ]:
def loss_per_image(
    eps, y, A, label, noise_labels, *,
    reg_type="none", reg_weight=0.0,
):
    """Total optimization objective for one image (batch size 1).

    With the default arguments, this is the original measurement loss.
    """
    assert eps.shape == (1, 256, 256, 3)
    assert noise_labels.shape == (1, model.noise_coords)

    class_id = jnp.asarray(label, dtype=jnp.int32).reshape(1)

    condition = model.apply(
        {"params": params},
        class_id,
        1.0,
        noise_labels,
        method=model.c_cfg_noise_to_cond,
    )

    x_hat = model.apply(
        {"params": params},
        eps,
        condition,
        True,
        method=model.generate_image,
    )

    predicted_y = A(x_hat.astype(jnp.float32))
    measured_y = jnp.asarray(y, dtype=jnp.float32)

    if predicted_y.shape != measured_y.shape:
        raise ValueError(
            f"Measurement shapes differ: {predicted_y.shape} "
            f"vs {measured_y.shape}"
        )

    measurement_loss = jnp.sum((predicted_y - measured_y) ** 2)
    penalty = eps_regularizer(eps, reg_type)

    # y.size converts the mean-sized penalty to the scale of the summed
    # measurement loss. Thus reg_weight is relative to measurement MSE.
    weighted_penalty = (
        jnp.asarray(reg_weight, dtype=measurement_loss.dtype)
        * measured_y.size
        * penalty
    )

    return measurement_loss + weighted_penalty


def loss_and_grad_eps(
    eps, y, A, label, noise_labels, *,
    reg_type="none", reg_weight=0.0,
):
    """Return the total objective and its gradient with respect to eps."""
    return jax.value_and_grad(loss_per_image, argnums=0)(
        eps, y, A, label, noise_labels,
        reg_type=reg_type,
        reg_weight=reg_weight,
    )


def gradient_step(
    eps, opt_state, y, A, label, noise_labels, optimizer, *,
    reg_type="none", reg_weight=0.0,
):
    """Take one optimizer step; return the objective before the update."""
    objective, grad_eps = loss_and_grad_eps(
        eps, y, A, label, noise_labels,
        reg_type=reg_type,
        reg_weight=reg_weight,
    )
    updates, new_opt_state = optimizer.update(
        grad_eps, opt_state, eps
    )
    new_eps = optax.apply_updates(eps, updates)
    return new_eps, new_opt_state, objective

### Conditional reconstruction 
Assums knowledge of the exact label of the image 
This lays the ground work for classifier -> conditional reconstruction pipeline

In [ ]:
# # Single-image Phase 1 smoke test
# rng = np.random.default_rng(2026)
# image_index = int(rng.integers(len(imageNet_x4)))

# with index_path.open(newline="") as file:
#     index_rows = list(csv.DictReader(file))

# row = index_rows[image_index]
# assert int(row["index"]) == image_index

# hr = load_hr256(Path(DATA_DIR) / row["relative_path"])
# label = int(row["label"])
# y = jnp.asarray(np.asarray(imageNet_x4[image_index:image_index + 1]))

# # Check that the saved measurement matches A_sr4.
# expected_y = np.asarray(jax.device_get(A_sr4(jnp.asarray(hr[None]))))
# np.testing.assert_allclose(expected_y, np.asarray(y), rtol=1e-5, atol=1e-5)

# key_eps, key_style = jax.random.split(jax.random.PRNGKey(2026))
# eps = jax.random.normal(key_eps, (1, 256, 256, 3), dtype=jnp.float32)

# eps_initial = eps  # NEW: JAX arrays are immutable, so this retains the initial eps.

# noise_labels = jax.random.randint(
#     key_style,
#     shape=(1, model.noise_coords),
#     minval=0,
#     maxval=model.noise_classes,
#     dtype=jnp.int32,
# )

# # NEW: ||eps|| / sqrt(d) equals the root mean square of its entries.
# def eps_norm_per_sqrt_d(eps):
#     eps = jnp.asarray(eps, dtype=jnp.float32)
#     return float(jnp.sqrt(jnp.mean(eps**2)))

# optimizer = optax.adam(learning_rate=0.02)
# opt_state = optimizer.init(eps)

# @jax.jit
# def train_step(eps, opt_state):
#     return gradient_step(
#         eps, opt_state, y, A_sr4, label, noise_labels, optimizer
#     )

# num_steps = 10_000  # NEW: one value controls the loop and report.
# log_every = 500      # NEW: avoid printing 10,000 lines.
# loss_history = []

# print(f"Image index: {image_index}, class label: {label}")
# print(f"Initial ‖ε‖/√d: {eps_norm_per_sqrt_d(eps_initial):.4f}")

# for step in range(num_steps):  # CHANGED: uses num_steps.
#     eps, opt_state, loss_before_step = train_step(eps, opt_state)
#     loss_value = float(jax.device_get(loss_before_step))
#     loss_history.append(loss_value)

#     if step == 0 or (step + 1) % log_every == 0:  # CHANGED
#         print(f"Before update {step + 1}: loss = {loss_value:.4f}")

# final_loss = float(jax.device_get(
#     loss_per_image(eps, y, A_sr4, label, noise_labels)
# ))
# loss_history.append(final_loss)

# print(f"After {num_steps} updates: loss = {final_loss:.4f}")
# print(f"Final   ‖ε‖/√d: {eps_norm_per_sqrt_d(eps):.4f}")  # NEW

# # Generate the final HR reconstruction.
# condition = model.apply(
#     {"params": params},
#     jnp.asarray([label], dtype=jnp.int32),
#     1.0,
#     noise_labels,
#     method=model.c_cfg_noise_to_cond,
# )
# x_hat = model.apply(
#     {"params": params},
#     eps,
#     condition,
#     True,
#     method=model.generate_image,
# )
# reconstruction = np.asarray(jax.device_get(x_hat), dtype=np.float32)[0]

# # NEW: Bicubic baseline. Upsample the observed LR image to HR size.
# # expected_y itself is 64×64, so it cannot be compared directly with hr.
# bicubic_hr = jax.image.resize(
#     y.astype(jnp.float32),
#     (1, *hr.shape),
#     method="cubic",
#     antialias=False,
# )
# bicubic_hr = np.asarray(jax.device_get(bicubic_hr), dtype=np.float32)[0]

# bicubic_psnr = psnr(hr, bicubic_hr)               # NEW
# reconstruction_psnr = psnr(hr, reconstruction)    # CHANGED

# print(f"Bicubic PSNR: {bicubic_psnr:.2f} dB")  # NEW
# print(f"Reconstruction PSNR after {num_steps} steps: "
#       f"{reconstruction_psnr:.2f} dB")         # CHANGED

# # CHANGED: add bicubic as a fifth panel.
# fig, axes = plt.subplots(1, 5, figsize=(20, 4))

# axes[0].imshow(np.clip((hr + 1) / 2, 0, 1))
# axes[0].set_title("Ground truth HR")

# axes[1].imshow(np.clip((np.asarray(y[0]) + 1) / 2, 0, 1))
# axes[1].set_title("Observed LR (×4)")

# axes[2].imshow(np.clip((bicubic_hr + 1) / 2, 0, 1))
# axes[2].set_title(f"Bicubic ({bicubic_psnr:.2f} dB)")

# axes[3].imshow(np.clip((reconstruction + 1) / 2, 0, 1))
# axes[3].set_title(f"Reconstruction ({reconstruction_psnr:.2f} dB)")

# axes[4].plot(range(len(loss_history)), loss_history)  # CHANGED: no markers for 10k points.
# axes[4].set_title("Measurement loss")
# axes[4].set_xlabel("Gradient steps")

# for ax in axes[:4]:
#     ax.axis("off")

# plt.tight_layout()
# plt.show()

In [ ]:
# # Single-image Phase 1 smoke test
# rng = np.random.default_rng(2026)  # reproducible random choice
# image_index = int(rng.integers(len(imageNet_x4)))

# with index_path.open(newline="") as file:
#     index_rows = list(csv.DictReader(file))

# row = index_rows[image_index]
# assert int(row["index"]) == image_index

# hr = load_hr256(Path(DATA_DIR) / row["relative_path"])
# label = int(row["label"])
# y = jnp.asarray(np.asarray(imageNet_x4[image_index:image_index + 1]))

# # Check that the saved measurement still matches our current A_sr4.
# expected_y = np.asarray(jax.device_get(A_sr4(jnp.asarray(hr[None]))))
# np.testing.assert_allclose(expected_y, np.asarray(y), rtol=1e-5, atol=1e-5)

# key_eps, key_style = jax.random.split(jax.random.PRNGKey(2026))
# eps = jax.random.normal(key_eps, (1, 256, 256, 3), dtype=jnp.float32)
# noise_labels = jax.random.randint(
#     key_style,
#     shape=(1, model.noise_coords),
#     minval=0,
#     maxval=model.noise_classes,
#     dtype=jnp.int32,
# )

# optimizer = optax.adam(learning_rate=0.02)
# opt_state = optimizer.init(eps)

# @jax.jit
# def train_step(eps, opt_state):
#     return gradient_step(
#         eps, opt_state, y, A_sr4, label, noise_labels, optimizer
#     )

# print(f"Image index: {image_index}, class label: {label}")
# loss_history = []

# for step in range(5):
#     eps, opt_state, loss_before_step = train_step(eps, opt_state)
#     loss_value = float(jax.device_get(loss_before_step))
#     loss_history.append(loss_value)
#     print(f"Step {step:2d}: measurement loss = {loss_value:.4f}")

# final_loss = float(jax.device_get(
#     loss_per_image(eps, y, A_sr4, label, noise_labels)
# ))
# loss_history.append(final_loss)
# print(f"Step  10000: measurement loss = {final_loss:.4f}")

# # Generate and inspect the final reconstructed HR image.
# condition = model.apply(
#     {"params": params},
#     jnp.asarray([label], dtype=jnp.int32),
#     1.0,
#     noise_labels,
#     method=model.c_cfg_noise_to_cond,
# )
# x_hat = model.apply(
#     {"params": params},
#     eps,
#     condition,
#     True,
#     method=model.generate_image,
# )
# reconstruction = np.asarray(jax.device_get(x_hat), dtype=np.float32)[0]

# print(f"HR PSNR after 5 steps: {psnr(hr, reconstruction):.2f} dB")

# fig, axes = plt.subplots(1, 4, figsize=(16, 4))
# axes[0].imshow(np.clip((hr + 1) / 2, 0, 1))
# axes[0].set_title("Ground truth HR")
# axes[1].imshow(np.clip((np.asarray(y[0]) + 1) / 2, 0, 1))
# axes[1].set_title("Observed LR (×4)")
# axes[2].imshow(np.clip((reconstruction + 1) / 2, 0, 1))
# axes[2].set_title("Reconstruction")
# axes[3].plot(range(len(loss_history)), loss_history, marker="o")
# axes[3].set_title("Measurement loss")
# axes[3].set_xlabel("Gradient steps")
# for ax in axes[:3]:
#     ax.axis("off")
# plt.tight_layout()
# plt.show()

### Random class reconstruction

In [ ]:
# # Single-image random-class baseline: no true label enters reconstruction.
# rng = np.random.default_rng(2026)
# image_index = int(rng.integers(len(imageNet_x4)))
# random_class = int(rng.integers(model.num_classes))

# y = jnp.asarray(
#     np.asarray(imageNet_x4[image_index:image_index + 1], dtype=np.float32)
# )

# key_eps, key_style = jax.random.split(jax.random.PRNGKey(2026))
# eps = jax.random.normal(key_eps, (1, 256, 256, 3), dtype=jnp.float32)
# noise_labels = jax.random.randint(
#     key_style,
#     shape=(1, model.noise_coords),
#     minval=0,
#     maxval=model.noise_classes,
#     dtype=jnp.int32,
# )

# optimizer = optax.adam(learning_rate=0.02)
# opt_state = optimizer.init(eps)

# @jax.jit
# def train_step(eps, opt_state):
#     return gradient_step(
#         eps, opt_state, y, A_sr4, random_class, noise_labels, optimizer
#     )

# num_steps = 10000
# log_every = 500
# history_steps = [0]
# history_loss = [
#     float(loss_per_image(eps, y, A_sr4, random_class, noise_labels))
# ]

# print(f"Image index: {image_index}")
# print(f"Random class used for reconstruction: {random_class}")
# print(f"Step 0: measurement loss = {history_loss[0]:.4f}")

# for step in range(1, num_steps + 1):
#     eps, opt_state, _ = train_step(eps, opt_state)

#     if step == 1 or step % log_every == 0 or step == num_steps:
#         current_loss = float(
#             loss_per_image(eps, y, A_sr4, random_class, noise_labels)
#         )
#         history_steps.append(step)
#         history_loss.append(current_loss)
#         print(f"Step {step}: measurement loss = {current_loss:.4f}")

# # Generate the reconstruction using the same random class.
# condition = model.apply(
#     {"params": params},
#     jnp.asarray([random_class], dtype=jnp.int32),
#     1.0,
#     noise_labels,
#     method=model.c_cfg_noise_to_cond,
# )
# x_hat = model.apply(
#     {"params": params},
#     eps,
#     condition,
#     True,
#     method=model.generate_image,
# )
# reconstruction = np.asarray(jax.device_get(x_hat), dtype=np.float32)[0]

# # Evaluation begins here: reveal the matching HR image and true label.
# with index_path.open(newline="") as file:
#     index_rows = list(csv.DictReader(file))

# row = index_rows[image_index]
# assert int(row["index"]) == image_index
# true_label = int(row["label"])
# hr = load_hr256(Path(DATA_DIR) / row["relative_path"])

# expected_y = np.asarray(jax.device_get(A_sr4(jnp.asarray(hr[None]))))
# np.testing.assert_allclose(expected_y, np.asarray(y), rtol=1e-5, atol=1e-5)

# print(f"True class (evaluation only): {true_label}")
# print("Model class:", random_class, "| True class:", true_label)
# print(f"HR PSNR after {num_steps} steps: {psnr(hr, reconstruction):.2f} dB")


# fig, axes = plt.subplots(1, 4, figsize=(16, 4))
# axes[0].imshow(np.clip((hr + 1) / 2, 0, 1))
# axes[0].set_title("Ground truth HR")
# axes[1].imshow(np.clip((np.asarray(y[0]) + 1) / 2, 0, 1))
# axes[1].set_title("Observed LR (×4)")
# axes[2].imshow(np.clip((reconstruction + 1) / 2, 0, 1))
# axes[2].set_title("Random-class reconstruction")
# axes[3].plot(history_steps, history_loss, marker="o")
# axes[3].set_title("Measurement loss")
# axes[3].set_xlabel("Gradient steps")
# for ax in axes[:3]:
#     ax.axis("off")
# plt.tight_layout()
# plt.show()

In [ ]:
# import json
# from urllib.request import urlopen

# url = (
#     "https://raw.githubusercontent.com/modestyachts/ImageNetV2/"
#     "master/data/metadata/class_info.json"
# )

# with urlopen(url) as response:
#     class_info = json.load(response)

# class_names = {
#     int(item["cid"]): item["synset"][0]
#     for item in class_info
# }

# print(f"Class fed to model: {class_names[random_class]} ({random_class})")
# print(f"Actual image class: {class_names[true_label]} ({true_label})")

### K FOLD CROSS VALIDATION

In [ ]:

# Stratified 80/20 split: 8 train and 2 test images per class.
with index_path.open(newline="") as file:
    index_rows = list(csv.DictReader(file))

index_rows.sort(key=lambda row: int(row["index"]))
assert [int(row["index"]) for row in index_rows] == list(range(len(imageNet_x4)))

labels = np.array([int(row["label"]) for row in index_rows])
class_ids, class_counts = np.unique(labels, return_counts=True)
assert len(class_ids) == 1_000
assert np.all(class_counts == 10), "Expected 10 images per class."

split_seed = 2026
split_rng = np.random.default_rng(split_seed)
train_indices = []
test_indices = []

for class_id in class_ids:
    class_indices = np.flatnonzero(labels == class_id)
    split_rng.shuffle(class_indices)

    test_indices.extend(class_indices[:2])
    train_indices.extend(class_indices[2:])

train_indices = np.asarray(train_indices, dtype=np.int32)
test_indices = np.asarray(test_indices, dtype=np.int32)

# Shuffle the order within each split, without changing membership.
split_rng.shuffle(train_indices)
split_rng.shuffle(test_indices)

# Verify size, balance, and absence of overlap.
assert len(train_indices) == 8_000
assert len(test_indices) == 2_000
assert np.intersect1d(train_indices, test_indices).size == 0
assert np.unique(np.concatenate([train_indices, test_indices])).size == len(labels)
assert np.all(np.bincount(labels[train_indices], minlength=1_000) == 8)
assert np.all(np.bincount(labels[test_indices], minlength=1_000) == 2)

split_path = Path(DATA_DIR) / "imageNetV2_split_80_20_seed2026.npz"
np.savez_compressed(
    split_path,
    train_indices=train_indices,
    test_indices=test_indices,
    seed=split_seed,
)

print(f"Train: {len(train_indices)} images, 8 per class")
print(f"Test:  {len(test_indices)} images, 2 per class")
print(f"Split saved to: {split_path}")

In [ ]:
def make_train_step(optimizer, reg_type):
    @jax.jit
    def train_step(eps, opt_state, y, label, noise_labels, reg_weight):
        return gradient_step(
            eps,
            opt_state,
            y,
            A_sr4,
            label,
            noise_labels,
            optimizer,
            reg_type=reg_type,
            reg_weight=reg_weight,
        )

    return train_step

In [ ]:
from itertools import product

learning_rates = (0.005, 0.02)
step_counts = (3_000,)
restart_counts = (2,)

regularization_options = (
    {"reg_type": "none",    "reg_weight": 0.0},
    {"reg_type": "moments", "reg_weight": 1.0},
    {"reg_type": "moments", "reg_weight": 10.0},
    {"reg_type": "moments", "reg_weight": 100.0},
)

base_grid = [
    {
        "learning_rate": lr,
        "num_steps": steps,
        "num_restarts": restarts,
    }
    for lr, steps, restarts in product(
        learning_rates, step_counts, restart_counts
    )
]

hyperparameter_grid = [
    {**base, **regularization}
    for base in base_grid
    for regularization in regularization_options
]

# One development image from each of 100 distinct classes.
# Keep the four-image pilot and the test set out of this search.
development_rng = np.random.default_rng(2027)
pilot_indices = np.asarray(train_indices[:4], dtype=np.int32)
pilot_classes = np.unique(labels[pilot_indices])
eligible_classes = np.setdiff1d(class_ids, pilot_classes)

selected_classes = development_rng.choice(
    eligible_classes, size=100, replace=False
)
sweep_indices = np.asarray(
    [
        int(development_rng.choice(
            train_indices[labels[train_indices] == class_id]
        ))
        for class_id in selected_classes
    ],
    dtype=np.int32,
)

assert len(sweep_indices) == 100
assert len(np.unique(labels[sweep_indices])) == 100
assert np.all(np.isin(sweep_indices, train_indices))
assert not np.any(np.isin(sweep_indices, test_indices))
assert not np.any(np.isin(sweep_indices, pilot_indices))

# Complete all configurations for an image before moving to the next.
run_plan = []
for image_index in sweep_indices:
    for config_id, config in enumerate(hyperparameter_grid):
        for restart in range(config["num_restarts"]):
            run_plan.append({
                "config_id": config_id,
                "image_index": int(image_index),
                "restart": restart,
                **config,
            })

reg_types = sorted({
    option["reg_type"] for option in regularization_options
})
step_cache = {}

for lr in learning_rates:
    optimizer = optax.adam(learning_rate=lr)
    for reg_type in reg_types:
        step_cache[(lr, reg_type)] = (
            optimizer,
            make_train_step(optimizer, reg_type),
        )

sweep_seed = 2026

def initialize_run(run):
    image_index = int(run["image_index"])

    y = jnp.asarray(
        np.asarray(
            imageNet_x4[image_index:image_index + 1],
            dtype=np.float32,
        )
    )

    # Oracle-class diagnostic: the true class is supplied to the model.
    label = jnp.asarray(
        int(index_rows[image_index]["label"]),
        dtype=jnp.int32,
    )

    # Matched initialization across configurations.
    key = jax.random.PRNGKey(sweep_seed)
    key = jax.random.fold_in(key, image_index)
    key = jax.random.fold_in(key, int(run["restart"]))
    key_eps, key_style = jax.random.split(key)

    eps = jax.random.normal(
        key_eps, (1, 256, 256, 3), dtype=jnp.float32
    )
    noise_labels = jax.random.randint(
        key_style,
        shape=(1, model.noise_coords),
        minval=0,
        maxval=model.noise_classes,
        dtype=jnp.int32,
    )

    optimizer, train_step = step_cache[
        (run["learning_rate"], run["reg_type"])
    ]
    opt_state = optimizer.init(eps)

    return y, label, noise_labels, eps, opt_state, train_step


first_run = run_plan[0]
y_run, label_run, style_run, eps_run, state_run, step_run = (
    initialize_run(first_run)
)

print("Configurations:", len(hyperparameter_grid))
print("Development images:", len(sweep_indices))
print("Distinct classes:", len(np.unique(labels[sweep_indices])))
print("Planned optimization runs:", len(run_plan))
print("Planned gradient steps:", sum(r["num_steps"] for r in run_plan))
print("First run:", first_run)
print("Shapes:", y_run.shape, eps_run.shape, style_run.shape)

#### Conditional reconstruction

In [ ]:
from time import perf_counter
from tqdm.auto import tqdm


def run_one_reconstruction(run, *, show_progress=False):
    y, label, noise_labels, eps, opt_state, train_step = initialize_run(run)
    initial_eps = eps

    # With default regularization arguments, this is measurement loss.
    initial_loss = float(
        loss_per_image(eps, y, A_sr4, label, noise_labels)
    )
    initial_penalty = float(eps_regularizer(eps, run["reg_type"]))
    penalty_scale = float(run["reg_weight"]) * y.size
    initial_objective = initial_loss + penalty_scale * initial_penalty

    start_time = perf_counter()
    sync_every = 50
    reported_steps = 0

    description = (
        f"Config {run['config_id']} | "
        f"image {run['image_index']} | "
        f"restart {run['restart']}"
    )

    with tqdm(
        total=run["num_steps"],
        desc=description,
        unit="step",
        disable=not show_progress,
        leave=False,
    ) as progress:
        for step in range(1, run["num_steps"] + 1):
            eps, opt_state, objective_before_step = train_step(
                eps,
                opt_state,
                y,
                label,
                noise_labels,
                run["reg_weight"],
            )

            if step % sync_every == 0 or step == run["num_steps"]:
                eps.block_until_ready()

                if show_progress:
                    progress.update(step - reported_steps)
                    progress.set_postfix(
                        objective_before=(
                            f"{float(objective_before_step):.4f}"
                        )
                    )
                    reported_steps = step

    final_loss = float(
        loss_per_image(eps, y, A_sr4, label, noise_labels)
    )
    final_penalty = float(eps_regularizer(eps, run["reg_type"]))
    final_objective = final_loss + penalty_scale * final_penalty

    condition = model.apply(
        {"params": params},
        jnp.asarray(label, dtype=jnp.int32).reshape(1),
        1.0,
        noise_labels,
        method=model.c_cfg_noise_to_cond,
    )
    x_hat = model.apply(
        {"params": params},
        eps,
        condition,
        True,
        method=model.generate_image,
    )
    reconstruction = np.asarray(
        jax.device_get(x_hat), dtype=np.float32
    )[0]

    row = index_rows[run["image_index"]]
    assert int(row["index"]) == run["image_index"]
    hr = load_hr256(Path(DATA_DIR) / row["relative_path"])

    return {
        **run,
        "initial_loss": initial_loss,
        "final_loss": final_loss,
        "initial_penalty": initial_penalty,
        "final_penalty": final_penalty,
        "initial_objective": initial_objective,
        "final_objective": final_objective,
        "psnr": psnr(hr, reconstruction),
        "initial_eps_rms": float(jnp.sqrt(jnp.mean(initial_eps**2))),
        "final_eps_rms": float(jnp.sqrt(jnp.mean(eps**2))),
        "initial_eps_mean": float(jnp.mean(initial_eps)),
        "final_eps_mean": float(jnp.mean(eps)),
        "initial_eps_std": float(jnp.std(initial_eps)),
        "final_eps_std": float(jnp.std(eps)),
        "elapsed_seconds": perf_counter() - start_time,
    }

In [ ]:
import fcntl
import hashlib
import json
import os
import tempfile
from pathlib import Path
from tqdm.auto import tqdm


# Change this version if you change the objective or reconstruction
# procedure while retaining the same run plan.
experiment_version = "sr4_oracle_moments_dev_v1"

experiment_spec = {
    "version": experiment_version,
    "model": "hf://pixel_B_sota",
    "operator": "A_sr4",
    "sweep_seed": sweep_seed,
    "run_plan": run_plan,
}

spec_text = json.dumps(
    experiment_spec,
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False,
)
plan_hash = hashlib.sha256(
    spec_text.encode("utf-8")
).hexdigest()[:12]

checkpoint_stem = (
    Path(DATA_DIR)
    / f"phase1_{experiment_version}_{plan_hash}"
)
results_path = Path(f"{checkpoint_stem}.jsonl")
manifest_path = Path(f"{checkpoint_stem}.manifest.json")
lock_path = Path(f"{checkpoint_stem}.lock")


def atomic_write_bytes(path, data):
    """Write a complete temporary file, then atomically replace path."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    fd, temporary_name = tempfile.mkstemp(
        prefix=f".{path.name}.",
        suffix=".tmp",
        dir=path.parent,
    )

    try:
        with os.fdopen(fd, "wb") as file:
            file.write(data)
            file.flush()
            os.fsync(file.fileno())

        os.replace(temporary_name, path)

        directory_fd = os.open(
            path.parent, os.O_RDONLY | os.O_DIRECTORY
        )
        try:
            os.fsync(directory_fd)
        finally:
            os.close(directory_fd)
    finally:
        if os.path.exists(temporary_name):
            os.unlink(temporary_name)


def atomic_save_results(path, rows):
    payload = "".join(
        json.dumps(
            row,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ) + "\n"
        for row in rows
    ).encode("utf-8")

    atomic_write_bytes(path, payload)


def load_results(path):
    rows = []
    with Path(path).open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    f"Invalid checkpoint line {line_number} in {path}"
                ) from error
    return rows


def run_key(run):
    return (
        int(run["config_id"]),
        int(run["image_index"]),
        int(run["restart"]),
    )


plan_by_key = {run_key(run): run for run in run_plan}
assert len(plan_by_key) == len(run_plan)

# A process that crashes releases this lock automatically.
with lock_path.open("a+") as lock_file:
    try:
        fcntl.flock(lock_file, fcntl.LOCK_EX | fcntl.LOCK_NB)
    except BlockingIOError as error:
        raise RuntimeError(
            "This checkpoint is already being written by another process."
        ) from error

    if manifest_path.exists():
        saved_spec = json.loads(
            manifest_path.read_text(encoding="utf-8")
        )
        if saved_spec != experiment_spec:
            raise RuntimeError(
                "Checkpoint manifest does not match the current run plan."
            )
    else:
        if results_path.exists():
            raise RuntimeError(
                "Results exist without a manifest; refusing to mix runs."
            )
        atomic_write_bytes(
            manifest_path,
            (spec_text + "\n").encode("utf-8"),
        )

    if results_path.exists():
        sweep_results = load_results(results_path)
    else:
        sweep_results = []
        atomic_save_results(results_path, sweep_results)

    # Check that every saved record belongs to this exact grid.
    for result in sweep_results:
        key = run_key(result)
        if key not in plan_by_key:
            raise RuntimeError(f"Unexpected saved run: {key}")

        planned = plan_by_key[key]
        for name in (
            "learning_rate",
            "num_steps",
            "num_restarts",
            "reg_type",
            "reg_weight",
        ):
            if result[name] != planned[name]:
                raise RuntimeError(
                    f"Saved run {key} has a different {name}"
                )

    completed = {run_key(result) for result in sweep_results}
    if len(completed) != len(sweep_results):
        
        raise RuntimeError("Duplicate runs in the checkpoint.")

    remaining_runs = [
        run for run in run_plan
        if run_key(run) not in completed
    ]

    # Process a short chunk, then run this cell again to continue.
    max_new_runs = len(remaining_runs)
    runs_this_call = remaining_runs[:max_new_runs]

    print(f"Checkpoint: {results_path}")
    print(f"Completed: {len(completed)} / {len(run_plan)}")
    print(f"Runs in this call: {len(runs_this_call)}")

    for run in tqdm(
        runs_this_call,
        desc="Regularization sweep",
        unit="run",
        mininterval=10,
    ):
        result = run_one_reconstruction(
            run, show_progress=False
        )

        # Write first; count the result only after saving succeeds.
        atomic_save_results(
            results_path,
            sweep_results + [result],
        )
        sweep_results.append(result)
        if len(sweep_results) % 10 == 0:
            tqdm.write(
                f"Checkpointed {len(sweep_results)}/{len(run_plan)} runs; "
                f"latest image={run['image_index']}, "
                f"config={run['config_id']}, restart={run['restart']}"
            )

    print(
        f"Saved {len(sweep_results)} / {len(run_plan)} "
        f"results to {results_path}"
    )

    if len(sweep_results) < len(run_plan):
        print("Run this cell again to continue.")

In [ ]:
# import csv
# import json
# from pathlib import Path

# # The checkpoint cell defines these and verifies the run plan.
# required = (
#     "results_path", "manifest_path", "experiment_spec",
#     "run_plan", "load_results", "run_key",
# )
# missing = [name for name in required if name not in globals()]
# if missing:
#     raise RuntimeError(f"Run the checkpoint cell (33) first: {missing}")

# saved_spec = json.loads(manifest_path.read_text(encoding="utf-8"))
# if saved_spec != experiment_spec:
#     raise RuntimeError("Checkpoint manifest differs from the current run plan.")

# results = load_results(results_path)
# if len(results) != len(run_plan):
#     raise RuntimeError(
#         f"Expected {len(run_plan)} completed runs, found {len(results)}"
#     )

# planned_keys = {run_key(run) for run in run_plan}
# saved_keys = [run_key(run) for run in results]
# if len(set(saved_keys)) != len(saved_keys) or set(saved_keys) != planned_keys:
#     raise RuntimeError("Checkpoint has missing, duplicate, or unexpected runs.")

# development_indices = sorted({
#     int(run["image_index"]) for run in run_plan
# })
# config_ids = sorted({
#     int(run["config_id"]) for run in run_plan
# })

# assert np.all(np.isin(development_indices, train_indices))
# assert not np.any(np.isin(development_indices, test_indices))

# with index_path.open(newline="") as file:
#     index_rows = list(csv.DictReader(file))
# index_rows.sort(key=lambda row: int(row["index"]))

# # Choose restarts using a quantity available at inference time.
# best_restart = {}
# for record in results:
#     if not np.isfinite(record["final_loss"]):
#         raise ValueError(f"Nonfinite measurement loss: {run_key(record)}")
#     if not np.isfinite(record["psnr"]):
#         raise ValueError(f"Nonfinite HR PSNR: {run_key(record)}")

#     key = (
#         int(record["config_id"]),
#         int(record["image_index"]),
#     )
#     if (
#         key not in best_restart
#         or record["final_loss"] < best_restart[key]["final_loss"]
#     ):
#         best_restart[key] = record

# if len(best_restart) != len(config_ids) * len(development_indices):
#     raise RuntimeError("Some configuration/image pairs are missing.")

# # Compute one bicubic baseline per development image.
# bicubic_psnr_by_image = {}
# for image_index in development_indices:
#     row = index_rows[image_index]
#     assert int(row["index"]) == image_index

#     hr = load_hr256(Path(DATA_DIR) / row["relative_path"])
#     y_image = jnp.asarray(
#         np.asarray(
#             imageNet_x4[image_index:image_index + 1],
#             dtype=np.float32,
#         )
#     )
#     bicubic_hr = jax.image.resize(
#         y_image,
#         (1, *hr.shape),
#         method="cubic",
#         antialias=False,
#     )
#     bicubic_hr = np.asarray(
#         jax.device_get(bicubic_hr), dtype=np.float32
#     )[0]
#     bicubic_psnr_by_image[image_index] = psnr(hr, bicubic_hr)

# mean_bicubic_psnr = float(np.mean(
#     list(bicubic_psnr_by_image.values())
# ))

# mean_final_loss = {}
# mean_hr_psnr = {}
# mean_gain_over_bicubic = {}
# images_beating_bicubic = {}

# for config_id in config_ids:
#     selected = [
#         best_restart[(config_id, image_index)]
#         for image_index in development_indices
#     ]

#     mean_final_loss[config_id] = float(np.mean([
#         run["final_loss"] for run in selected
#     ]))
#     mean_hr_psnr[config_id] = float(np.mean([
#         run["psnr"] for run in selected
#     ]))

#     gains = [
#         best_restart[(config_id, image_index)]["psnr"]
#         - bicubic_psnr_by_image[image_index]
#         for image_index in development_indices
#     ]
#     mean_gain_over_bicubic[config_id] = float(np.mean(gains))
#     images_beating_bicubic[config_id] = sum(
#         gain > 0 for gain in gains
#     )

# # Rank by development HR PSNR; use measurement loss to break ties.
# ranking = sorted(
#     config_ids,
#     key=lambda config_id: (
#         -mean_hr_psnr[config_id],
#         mean_final_loss[config_id],
#         config_id,
#     ),
# )
# best_config_id = ranking[0]
# best_loss_config_id = min(
#     config_ids, key=lambda config_id: mean_final_loss[config_id]
# )

# print(f"Development images: {len(development_indices)}")
# print(f"Mean bicubic HR PSNR: {mean_bicubic_psnr:.2f} dB")
# print("\nConfigurations ranked by mean development HR PSNR:")

# for config_id in ranking:
#     example = best_restart[
#         (config_id, development_indices[0])
#     ]
#     print(
#         f"Config {config_id:2d}: "
#         f"HR PSNR={mean_hr_psnr[config_id]:5.2f} dB | "
#         f"vs bicubic={mean_gain_over_bicubic[config_id]:+5.2f} dB | "
#         f"wins={images_beating_bicubic[config_id]}/"
#         f"{len(development_indices)} | "
#         f"loss={mean_final_loss[config_id]:8.2f} | "
#         f"lr={example['learning_rate']}, "
#         f"steps={example['num_steps']}, "
#         f"restarts={example['num_restarts']}, "
#         f"reg={example['reg_type']}, "
#         f"weight={example['reg_weight']}"
#     )

# print(
#     f"\nSelected by HR PSNR: config {best_config_id} "
#     f"({mean_hr_psnr[best_config_id]:.2f} dB)"
# )
# print(
#     f"Lowest measurement loss: config {best_loss_config_id} "
#     f"({mean_final_loss[best_loss_config_id]:.2f})"
# )

# # Choose a development image reproducibly, without selecting by PSNR.
# image_index = int(
#     np.random.default_rng(2026).choice(development_indices)
# )
# best_run = best_restart[(best_config_id, image_index)]

# print(
#     f"\nDisplaying image {image_index}, "
#     f"config {best_config_id}, restart {best_run['restart']}, "
#     f"regularizer {best_run['reg_type']} "
#     f"at weight {best_run['reg_weight']}"
# )

# # Recreate the run because the checkpoint stores metrics, not images or eps.
# y, label, noise_labels, eps, opt_state, train_step = (
#     initialize_run(best_run)
# )

# num_steps = int(best_run["num_steps"])
# reported_steps = 0

# with tqdm(
#     total=num_steps,
#     desc="Recreating selected reconstruction",
#     unit="step",
# ) as progress:
#     for step in range(1, num_steps + 1):
#         eps, opt_state, _ = train_step(
#             eps,
#             opt_state,
#             y,
#             label,
#             noise_labels,
#             best_run["reg_weight"],
#         )

#         if step % 50 == 0 or step == num_steps:
#             eps.block_until_ready()
#             progress.update(step - reported_steps)
#             reported_steps = step

# condition = model.apply(
#     {"params": params},
#     jnp.asarray(label, dtype=jnp.int32).reshape(1),
#     1.0,
#     noise_labels,
#     method=model.c_cfg_noise_to_cond,
# )
# x_hat = model.apply(
#     {"params": params},
#     eps,
#     condition,
#     True,
#     method=model.generate_image,
# )

# reconstruction = np.asarray(
#     jax.device_get(x_hat), dtype=np.float32
# )[0]
# predicted_lr = np.asarray(
#     jax.device_get(A_sr4(x_hat.astype(jnp.float32))),
#     dtype=np.float32,
# )[0]
# observed_lr = np.asarray(
#     jax.device_get(y), dtype=np.float32
# )[0]

# row = index_rows[image_index]
# hr = load_hr256(Path(DATA_DIR) / row["relative_path"])

# bicubic_hr = jax.image.resize(
#     y.astype(jnp.float32),
#     (1, *hr.shape),
#     method="cubic",
#     antialias=False,
# )
# bicubic_hr = np.asarray(
#     jax.device_get(bicubic_hr), dtype=np.float32
# )[0]

# bicubic_psnr = psnr(hr, bicubic_hr)
# reconstruction_psnr = psnr(hr, reconstruction)
# lr_fit_psnr = psnr(observed_lr, predicted_lr)
# measurement_loss = float(
#     np.sum((predicted_lr - observed_lr) ** 2)
# )
# penalty = float(
#     eps_regularizer(eps, best_run["reg_type"])
# )
# total_objective = (
#     measurement_loss
#     + best_run["reg_weight"] * observed_lr.size * penalty
# )

# print(f"\nBicubic HR PSNR:        {bicubic_psnr:.2f} dB")
# print(f"Reconstruction HR PSNR: {reconstruction_psnr:.2f} dB")
# print(
#     f"Difference vs bicubic:  "
#     f"{reconstruction_psnr - bicubic_psnr:+.2f} dB"
# )
# print(f"LR measurement PSNR:    {lr_fit_psnr:.2f} dB")
# print(f"Measurement loss:       {measurement_loss:.2f}")
# print(f"Raw ε penalty:          {penalty:.6f}")
# print(f"Total objective:        {total_objective:.2f}")
# print(f"Saved HR PSNR:          {best_run['psnr']:.2f} dB")
# print(f"Saved measurement loss: {best_run['final_loss']:.2f}")

# # Clipping affects display only; PSNR uses the original arrays.
# def display_rgb(image):
#     return np.clip((image + 1.0) / 2.0, 0.0, 1.0)

# fig, axes = plt.subplots(1, 5, figsize=(20, 4))

# panels = [
#     (hr, "Ground truth HR"),
#     (observed_lr, "Observed LR (64×64)"),
#     (bicubic_hr, f"Bicubic HR\n{bicubic_psnr:.2f} dB"),
#     (
#         reconstruction,
#         f"Reconstructed HR\n{reconstruction_psnr:.2f} dB",
#     ),
#     (
#         predicted_lr,
#         f"Reconstructed LR\nfit: {lr_fit_psnr:.2f} dB",
#     ),
# ]

# for ax, (image, title) in zip(axes, panels):
#     ax.imshow(display_rgb(image), interpolation="nearest")
#     ax.set_title(title)
#     ax.axis("off")

# plt.tight_layout()
# plt.show()

In [ ]:
best_lr = 0.02
best_iterations = 3000
best_regul = None
best_restarts = 2